In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import xgboost as xgb
from scipy import stats
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split, RandomizedSearchCV, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.utils.class_weight import compute_class_weight
from sklearn.datasets import make_classification
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
import pickle

In [ ]:
# read in data - this is a CSV file of general features for around 730K stars
data_path = "ALL_full_star_table_all_types.txt"
data = pd.read_csv(data_path, sep='\t', skiprows=6, header=0)

In [ ]:
data

In [ ]:
data.rename(columns={'# V_I': 'V_I'}, inplace=True)

In [ ]:
labels = data["Type"]

In [ ]:
np.unique(labels)

In [ ]:
data.head()

In [ ]:
data.shape

In [ ]:
labels

In [ ]:
# counts of each variable star type
type_counts = data['Type'].value_counts()
type_counts, type_counts / type_counts.sum()

In [ ]:
# replacing -99.99s with NaNs
data = data.replace(-99.99, np.nan)
data = data.replace("-99.99", np.nan)

In [ ]:
high_nan = data.columns[data.isna().mean() > 0.2]

# create a DataFrame with non-NaN columns
df_filtered = data.drop(columns=high_nan)
df_filtered.head()

In [ ]:
non_numeric_columns = df_filtered.select_dtypes(exclude=['number'])
df_numeric = df_filtered.drop(columns=non_numeric_columns)
df_numeric

In [ ]:
features = ['T0_1', 'A_1', 'I', 'P_1']
data = df_numeric[features]

In [ ]:
data

In [ ]:
# dropping NaNs from dataframe
nan_indices = data[data.isna().any(axis=1)].index
data = data.dropna()

In [ ]:
data

In [ ]:
# removing corresponding indices from labels
labels = [label for i, label in enumerate(labels) if i not in nan_indices]
classes = np.unique(labels)
data = data.reset_index(drop=True)

In [ ]:
data

In [ ]:
classes

In [ ]:
data.shape, len(labels)

In [ ]:
encoder = LabelEncoder()
labels_encoded = encoder.fit_transform(labels)
labels_encoded

In [ ]:
# smote instance 
smote = SMOTE(sampling_strategy='auto', random_state=21)
X_resampled, y_resampled = smote.fit_resample(data, labels_encoded)

In [ ]:
# Count the class distribution before and after SMOTE
print("Class distribution before SMOTE:\n", pd.Series(labels).value_counts())
print("Class distribution after SMOTE:\n", pd.Series(y_resampled).value_counts())

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(X_resampled, 
                                                    y_resampled, 
                                                    test_size=0.3, 
                                                    random_state=21)

X_val, X_test, y_val, y_test = train_test_split(X_temp, 
                                                y_temp, 
                                                test_size=0.5, 
                                                random_state=21)


In [ ]:
[len(dataset) for dataset in [X_train, y_train, X_val, X_test, y_val, y_test]]

In [ ]:
# hyperparameter tuning for random forest
param_dist = {
    "n_estimators": [100, 200, 300, 400], 
    "max_depth": [None, 10, 20, 30], 
    "min_samples_split": [2, 5, 10, 15], 
    "min_samples_leaf": [1, 2, 4, 8]
}

rf_tuning = RandomForestClassifier(random_state=21)

rand_search = RandomizedSearchCV(estimator=rf_tuning, 
                                 param_distributions=param_dist, 
                                 n_iter=20, 
                                 cv=5, 
                                 scoring="accuracy", 
                                 random_state=21, 
                                 n_jobs=-1)

rand_search.fit(X_val, y_val)

best_params = rand_search.best_params_
best_params

In [ ]:
rf = RandomForestClassifier(n_estimators=400, 
                            min_samples_split=2, 
                            min_samples_leaf=1, 
                            max_depth=30,
                            random_state=21)
rf.fit(X_train, y_train)

In [ ]:
with open('RF_model.pkl','wb') as f:
     pickle.dump(rf, f)

In [ ]:
predictions = rf.predict(X_test)

accuracy = accuracy_score(y_test, predictions)
accuracy * 100

In [ ]:
conf_mat = confusion_matrix(y_test, predictions)
conf_mat_pct = conf_mat.astype("float") / conf_mat.sum(axis=1)[:, np.newaxis] * 100

plt.figure(figsize=(16, 14))
ax = sns.heatmap(conf_mat_pct, fmt=".2f", cmap="Blues", cbar=False,
            xticklabels=classes,  # Predicted
            yticklabels=classes)  

for i in range(len(classes)):
    for j in range(len(classes)):
        count = conf_mat[i, j]
        percent = conf_mat_pct[i, j]
        text = f"{count} ({percent:.2f}%)"
        color = 'white' if i == j else 'black'  # White for diagonal, black for others
        ax.text(j + 0.5, i + 0.5, text, ha='center', va='center', color=color, fontsize=14)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Random Forest Confusion Matrix")
plt.show()

In [ ]:
import pickle

In [ ]:
with open('conf_mat_2_RF.pkl','wb') as f:
     pickle.dump(conf_mat, f)

In [ ]:
with open('conf_mat_pct_RF.pkl','wb') as f:
     pickle.dump(conf_mat_pct, f)